# Aula 21 — Planning and Goal Decomposition

**Quando escolher apenas a próxima ação deixa de ser suficiente?**

Na Aula 20, o agentic loop escolhia uma ação por vez a partir do estado observado.

Agora adicionamos uma nova capacidade:

> **construir, validar e revisar um plano explícito de múltiplas etapas.**

Regra da aula:

```text
planejar
≠
pensar mais

planejar
=
representar subobjetivos
+ dependências
+ pré-condições
+ ordem
+ revisão sob evidência
```

## 1. Objetivos

Ao final, você deverá conseguir:

- distinguir reactive next-action de planning explícito;
- decompor um objetivo em subtarefas;
- representar dependências;
- validar um plano antes da execução;
- separar plan generation de plan execution;
- observar replanning;
- medir overhead de planejamento;
- reconhecer quando o plano não traz utility adicional.

## 📘 Glossário da aula

- **[Planejamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#planejamento)**
- **[Decomposição de objetivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#decomposição-de-objetivo)**
- **[Replanejamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#replanejamento)**
- **[Validação de plano](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#validação-de-plano)**

## 2. Reactive loop vs explicit planning

```text
Reactive
state → next action → execute → observe → repeat

Planning
goal → decompose → build plan → validate
     → execute → observe → continue / replan
```

Um plano não é “raciocínio escondido”.

Nesta aula ele será um **artefato estruturado e observável**.

### 2.1 Goal decomposition — antes do plano

Um objetivo amplo não é ainda um plano.

Nesta aula usamos:

```text
goal
→ subgoals
→ plan steps
→ execution
```

Exemplo:

```text
Goal:
publicar uma aula com segurança

Subgoals:
1. validar o pedido
2. confirmar prontidão
3. preparar release
4. obter autorização
5. publicar
```

A decomposição responde:

> **Que resultados intermediários precisam existir para que o objetivo maior possa ser alcançado?**

O plano responde outra pergunta:

> **Em que ordem, sob quais dependências e pré-condições esses subobjetivos serão executados?**

Portanto:

```text
goal decomposition
≠
plan execution
```

In [1]:
from dataclasses import dataclass, field
import pandas as pd

ACTIONS = {
    "inspect_request",
    "get_lesson_status",
    "build_release_note",
    "request_approval",
    "publish_simulated",
    "ask_human",
    "stop",
}

SCENARIOS = {
    "simple_ready": {
        "request_valid": True,
        "initial_lesson_status": "ready",
        "approval_available": True,
        "status_changes_after_first_read": False,
    },
    "not_ready": {
        "request_valid": True,
        "initial_lesson_status": "review-candidate",
        "approval_available": True,
        "status_changes_after_first_read": False,
    },
    "state_changes": {
        "request_valid": True,
        "initial_lesson_status": "ready",
        "changed_lesson_status": "review-candidate",
        "approval_available": True,
        "status_changes_after_first_read": True,
    },
}

print("Cenários:", list(SCENARIOS))

Cenários: ['simple_ready', 'not_ready', 'state_changes']


In [2]:
GOAL = "publicar uma aula com segurança"

SUBGOALS = [
    {"id": "g1", "subgoal": "validar o pedido"},
    {"id": "g2", "subgoal": "confirmar prontidão"},
    {"id": "g3", "subgoal": "preparar release"},
    {"id": "g4", "subgoal": "obter autorização"},
    {"id": "g5", "subgoal": "publicar"},
]

print("Goal:", GOAL)
display(pd.DataFrame(SUBGOALS))

Goal: publicar uma aula com segurança


,id,subgoal
0,g1,validar o pedido
1,g2,confirmar prontidão
2,g3,preparar release
3,g4,obter autorização
4,g5,publicar


## 3. Contrato do plano

Cada passo terá:

```text
step_id
subgoal
action
depends_on
preconditions
status
reason
```

Isso torna o plano um artefato inspecionável.

- `depends_on` representa dependências estruturais entre etapas;
- `preconditions` representa fatos que precisam continuar verdadeiros para uma ação ser válida;
- `status` permite acompanhar o ciclo de vida do passo;
- `reason` registra a justificativa operacional resumida.

Uma dependência e uma pré-condição não são a mesma coisa:

```text
depends_on
→ "esta etapa deve ocorrer depois daquela"

precondition
→ "este fato precisa ser verdadeiro agora"
```

In [3]:
@dataclass
class PlanStep:
    step_id: str
    action: str
    subgoal: str = ""
    depends_on: list = field(default_factory=list)
    preconditions: list = field(default_factory=list)
    status: str = "pending"
    reason: str = ""

def build_plan():
    return [
        PlanStep(
            "s1", "inspect_request",
            subgoal="validar o pedido",
            preconditions=[],
            reason="validar pedido",
        ),
        PlanStep(
            "s2", "get_lesson_status",
            subgoal="confirmar prontidão",
            depends_on=["s1"],
            preconditions=["request_valid == True"],
            reason="observar prontidão",
        ),
        PlanStep(
            "s3", "build_release_note",
            subgoal="preparar release",
            depends_on=["s2"],
            preconditions=["lesson_status == ready"],
            reason="preparar release",
        ),
        PlanStep(
            "s4", "request_approval",
            subgoal="obter autorização",
            depends_on=["s3"],
            preconditions=["release_note exists"],
            reason="obter autorização",
        ),
        PlanStep(
            "s5", "publish_simulated",
            subgoal="publicar",
            depends_on=["s4"],
            preconditions=["approval == True", "lesson_status == ready"],
            reason="publicar com aprovação",
        ),
    ]

plan = build_plan()
display(pd.DataFrame([vars(x) for x in plan]))

,step_id,action,subgoal,depends_on,preconditions,status,reason
0,s1,inspect_request,validar o pedido,[],[],pending,validar pedido
1,s2,get_lesson_status,confirmar prontidão,[s1],[request_valid == True],pending,observar prontidão
2,s3,build_release_note,preparar release,[s2],[lesson_status == ready],pending,preparar release
3,s4,request_approval,obter autorização,[s3],[release_note exists],pending,obter autorização
4,s5,publish_simulated,publicar,[s4],"[approval == True, lesson_status == ready]",pending,publicar com aprovação


## 4. Validação de plano

Antes da execução, validamos o artefato.

Critérios mínimos:

- action na allow-list;
- dependências existentes;
- nenhuma dependência aponta para o futuro;
- publicação depende de aprovação.

In [4]:
def validate_plan(plan):
    ids = [s.step_id for s in plan]
    id_to_pos = {step_id: i for i, step_id in enumerate(ids)}
    action_to_pos = {step.action: i for i, step in enumerate(plan)}
    errors = []

    # IDs duplicados tornam dependências ambíguas.
    if len(ids) != len(set(ids)):
        errors.append(("plan", "duplicate_step_id"))

    for i, step in enumerate(plan):
        if step.action not in ACTIONS:
            errors.append((step.step_id, "action_not_allowed"))

        for dep in step.depends_on:
            if dep not in id_to_pos:
                errors.append((step.step_id, f"missing_dependency:{dep}"))
            elif id_to_pos[dep] >= i:
                errors.append((step.step_id, f"dependency_not_completed_before:{dep}"))

        if step.action == "publish_simulated":
            approval_pos = action_to_pos.get("request_approval")
            if approval_pos is None:
                errors.append((step.step_id, "approval_step_missing"))
            elif approval_pos >= i:
                errors.append((step.step_id, "approval_must_precede_publish"))

    return errors

print("Plan errors:", validate_plan(plan))

Plan errors: []


### 4.1 Failure Lab — plano inválido

Vamos colocar `publish_simulated` **antes** de `request_approval`.

O objetivo é verificar se a validação estrutural detecta a violação **antes que qualquer ação seja executada**.

Essa distinção será importante ao longo da aula:

```text
plan generation
→ produz uma proposta de plano

plan validation
→ verifica se a estrutura é admissível

plan execution
→ só começa depois da validação
```

In [5]:
bad_plan = [
    PlanStep("s1", "inspect_request", subgoal="validar o pedido"),
    PlanStep("s2", "get_lesson_status", subgoal="confirmar prontidão", depends_on=["s1"]),
    PlanStep(
        "s3", "publish_simulated",
        subgoal="publicar",
        depends_on=["s2"],
        preconditions=["approval == True"],
    ),
    PlanStep(
        "s4", "request_approval",
        subgoal="obter autorização",
        depends_on=["s3"],
    ),
]

display(pd.DataFrame([vars(x) for x in bad_plan]))
print("Plan errors:", validate_plan(bad_plan))

,step_id,action,subgoal,depends_on,preconditions,status,reason
0,s1,inspect_request,validar o pedido,[],[],pending,
1,s2,get_lesson_status,confirmar prontidão,[s1],[],pending,
2,s3,publish_simulated,publicar,[s2],[approval == True],pending,
3,s4,request_approval,obter autorização,[s3],[],pending,


Plan errors: [('s3', 'approval_must_precede_publish')]


## 5. Baseline reativo

Reutilizamos o princípio da Aula 20: uma decisão por vez, sem plano persistente.

Para manter a comparação justa, o baseline **não recebe informação privilegiada**. Ele começa sem ter observado `request_valid` nem `lesson_status`, e enfrenta a mesma dinâmica externa do executor plan-based.

No cenário `state_changes`, o ambiente muda depois da primeira leitura do status. Antes da ação sensível de publicação, o baseline relê a pré-condição externa.

In [6]:
def make_environment(scenario):
    return {
        "request_valid": scenario["request_valid"],
        "lesson_status": scenario["initial_lesson_status"],
        "approval_available": scenario["approval_available"],
        "status_changes_after_first_read": scenario.get("status_changes_after_first_read", False),
        "changed_lesson_status": scenario.get("changed_lesson_status"),
        "status_reads": 0,
    }

def read_lesson_status(env):
    value = env["lesson_status"]
    env["status_reads"] += 1

    # A mudança ocorre no ambiente após a primeira leitura.
    # Ambas as arquiteturas recebem exatamente a mesma dinâmica externa.
    if env["status_changes_after_first_read"] and env["status_reads"] == 1:
        env["lesson_status"] = env["changed_lesson_status"]

    return value

def run_reactive(scenario):
    env = make_environment(scenario)
    state = {
        "request_valid": None,
        "lesson_status": None,
        "approval": None,
        "published": False,
        "needs_human": False,
        "release_note": None,
    }
    trace = []

    while True:
        if state["request_valid"] is None:
            action = "inspect_request"
        elif state["request_valid"] is not True:
            action = "ask_human"
        elif state["lesson_status"] is None:
            action = "get_lesson_status"
        elif state["lesson_status"] != "ready":
            action = "stop"
        elif state["release_note"] is None:
            action = "build_release_note"
        elif state["approval"] is None:
            action = "request_approval"
        elif state["approval"] is not True:
            action = "ask_human"
        elif not state["published"]:
            # Antes da ação irreversível, o baseline reativo relê a pré-condição externa.
            latest_status = read_lesson_status(env)
            if latest_status != "ready":
                state["lesson_status"] = latest_status
                trace.append("refresh_lesson_status")
                trace.append("stop")
                break
            action = "publish_simulated"
        else:
            action = "stop"

        if action == "inspect_request":
            state["request_valid"] = env["request_valid"]
        elif action == "get_lesson_status":
            state["lesson_status"] = read_lesson_status(env)
        elif action == "build_release_note":
            state["release_note"] = "release simulada"
        elif action == "request_approval":
            state["approval"] = env["approval_available"]
        elif action == "publish_simulated":
            state["published"] = True
        elif action == "ask_human":
            state["needs_human"] = True
            trace.append(action)
            break
        elif action == "stop":
            trace.append(action)
            break

        trace.append(action)

    return state, trace

## 6. Execução orientada a plano

O executor percorre um plano previamente validado, mas não deve assumir que o mundo permaneceu congelado desde a criação do plano.

Antes de uma ação sensível, ele revalida uma pré-condição externa.

Se a evidência atual contradiz o plano:

```text
plano v1
+ nova observação incompatível
→ plano ficou stale
→ gerar plano v2
→ validar plano v2
→ executar recuperação segura
```

Nesta aula, o replanning é propositalmente pequeno: quando o status deixa de ser `ready`, o novo plano abandona a publicação e produz um `stop` seguro.

Isso já é diferente de apenas escrever “replan” no log: há um **novo artefato de plano**, armazenado em `plan_history`.

In [7]:
def build_recovery_plan(state):
    """Plano mínimo de recuperação quando o plano original fica stale."""
    if state["lesson_status"] != "ready":
        return [
            PlanStep(
                "r1",
                "stop",
                subgoal="interromper publicação insegura",
                preconditions=["lesson_status != ready"],
                reason="status atual invalida o plano de publicação",
            )
        ]

    return [
        PlanStep(
            "r1",
            "ask_human",
            subgoal="solicitar decisão externa",
            reason="estado inesperado exige supervisão",
        )
    ]

def run_plan_based(scenario):
    plan = build_plan()
    errors = validate_plan(plan)
    if errors:
        return {"published": False, "needs_human": False}, [], 0, errors, [plan]

    env = make_environment(scenario)
    state = {
        "request_valid": None,
        "lesson_status": None,
        "approval": None,
        "published": False,
        "needs_human": False,
        "release_note": None,
    }

    trace = []
    replans = 0
    plan_history = [plan]

    for step in plan:
        action = step.action

        if action == "inspect_request":
            state["request_valid"] = env["request_valid"]
            trace.append(action)
            if state["request_valid"] is not True:
                state["needs_human"] = True
                trace.append("ask_human")
                break
            continue

        if action == "get_lesson_status":
            state["lesson_status"] = read_lesson_status(env)
            trace.append(action)
            if state["lesson_status"] != "ready":
                trace.append("stop")
                break
            continue

        if action == "build_release_note":
            state["release_note"] = "release simulada"
            trace.append(action)
            continue

        if action == "request_approval":
            state["approval"] = env["approval_available"]
            trace.append(action)
            if state["approval"] is not True:
                state["needs_human"] = True
                trace.append("ask_human")
                break
            continue

        if action == "publish_simulated":
            # Revalida uma pré-condição externa antes da ação sensível.
            latest_status = read_lesson_status(env)

            if latest_status != state["lesson_status"]:
                state["lesson_status"] = latest_status
                replans += 1

                revised_plan = build_recovery_plan(state)
                revised_errors = validate_plan(revised_plan)
                plan_history.append(revised_plan)

                trace.append("replan:v2")

                if revised_errors:
                    state["needs_human"] = True
                    trace.append("ask_human")
                    break

                recovery_action = revised_plan[0].action
                trace.append(recovery_action)

                if recovery_action == "ask_human":
                    state["needs_human"] = True

                break

            state["published"] = True
            trace.append(action)

    return state, trace, replans, errors, plan_history

## 7. Comparative Evidence Lab

Agora executamos as duas arquiteturas sobre os mesmos cenários.

In [8]:
rows = []

for name, scenario in SCENARIOS.items():
    r_state, r_trace = run_reactive(scenario)
    rows.append({
        "architecture": "reactive",
        "scenario": name,
        "success": r_state["published"],
        "actions": len(r_trace),
        "planning_steps": 0,
        "plan_versions": 0,
        "replans": 0,
        "human_intervention": r_state["needs_human"],
        "cost_proxy": len(r_trace),
    })

    p_state, p_trace, replans, errors, plan_history = run_plan_based(scenario)
    rows.append({
        "architecture": "plan_based",
        "scenario": name,
        "success": p_state["published"],
        "actions": len(p_trace),
        "planning_steps": sum(len(p) for p in plan_history),
        "plan_versions": len(plan_history),
        "replans": replans,
        "human_intervention": p_state["needs_human"],
        "cost_proxy": len(p_trace) + 0.5 * sum(len(p) for p in plan_history) + replans,
    })

comparison = pd.DataFrame(rows)
display(comparison)

,architecture,scenario,success,actions,planning_steps,plan_versions,replans,human_intervention,cost_proxy
0,reactive,simple_ready,True,6,0,0,0,False,6.0
1,plan_based,simple_ready,True,5,5,1,0,False,7.5
2,reactive,not_ready,False,3,0,0,0,False,3.0
3,plan_based,not_ready,False,3,5,1,0,False,5.5
4,reactive,state_changes,False,6,0,0,0,False,6.0
5,plan_based,state_changes,False,6,6,2,1,False,10.0


### Como interpretar a comparação

Procure três situações:

1. **mesmo resultado + overhead de planejamento**  
   → planning não conquistou sua complexidade;

2. **erro estrutural rejeitado antes da execução**  
   → plan validation trouxe valor;

3. **mudança externa invalida uma pré-condição**  
   → o plano v1 fica stale e um plano v2 de recuperação pode impedir execução baseada em estado antigo.

A coluna `plan_versions` torna o replanning observável:

```text
1
→ apenas plano original

2
→ plano original + plano revisado
```

A comparação usa a mesma dinâmica de ambiente para as duas arquiteturas. Portanto, qualquer diferença deve vir do mecanismo arquitetural, não de informação privilegiada fornecida a uma delas.

> **Planning deve melhorar coordenação, segurança ou capacidade de recuperação de forma observável. Caso contrário, a arquitetura mais simples continua preferível.**

## 8. Architecture Decision Lab

Escolha entre:

```text
reactive next-action
explicit planning
deterministic workflow
```

Casos:

- A: sequência curta e estável;
- B: muitas dependências conhecidas;
- C: objetivo muda pouco, mas estado pode invalidar etapas futuras;
- D: tarefa de alto risco com ações irreversíveis.

Justifique por:

- variabilidade;
- dependências;
- reversibilidade;
- necessidade de inspeção prévia;
- custo de planning;
- risco de stale plan.

In [9]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução de referência\n\n{self._solution_text}"))

q21_arch = TILExercise(
    "Pergunte se há dependências suficientes para justificar um plano explícito e se o estado pode invalidar etapas futuras.",
    "A tende a workflow/reactive simples; B favorece planning; C favorece planning com validação/replanning; D requer planning governado e forte supervisão humana.",
)

print("Use q21_arch.hint() ou q21_arch.solution() somente quando desejar.")

Use q21_arch.hint() ou q21_arch.solution() somente quando desejar.


## 9. Checagem de compreensão

Responda antes de abrir as respostas de referência.

### Conceitos

1. Qual a diferença entre goal decomposition e plan generation?
2. Qual a diferença entre `depends_on` e uma `precondition`?
3. Qual a diferença entre plan generation e plan execution?
4. Por que um plano deve ser validado antes de executar?
5. Por que um plano explícito não é chain-of-thought?

### Evidência

6. Que falha o Failure Lab detecta antes de qualquer tool call?
7. O que significa dizer que um plano ficou `stale`?
8. No cenário `state_changes`, o que caracteriza replanning real nesta versão?
9. Por que as duas arquiteturas precisam receber a mesma dinâmica externa?
10. O que `plan_versions=2` nos permite observar?

### Decisão arquitetural

11. Em que cenário planning pode adicionar apenas overhead?
12. Quando um workflow determinístico ainda é melhor?
13. Quando um reactive loop pode ser suficiente sem plano persistente?
14. Que evidência justificaria adotar planning?
15. Como a Aula 21 estende o princípio “complexidade precisa ser conquistada por evidência”?

### Respostas de referência — revele depois de responder

<details>
<summary><strong>1. Goal decomposition vs plan generation</strong></summary>

Goal decomposition transforma um objetivo amplo em subobjetivos intermediários. Plan generation transforma esses subobjetivos em uma estrutura executável com ações, ordem, dependências e pré-condições.

</details>

<details>
<summary><strong>2. depends_on vs precondition</strong></summary>

`depends_on` expressa relação estrutural entre passos: uma etapa precisa vir depois de outra. `precondition` expressa um fato que precisa estar verdadeiro no momento da execução.

</details>

<details>
<summary><strong>3. Plan generation vs plan execution</strong></summary>

Generation produz o artefato de plano. Execution percorre um plano já validado, aplica suas ações e observa resultados.

</details>

<details>
<summary><strong>4. Por que validar antes?</strong></summary>

Para rejeitar erros estruturais, ações não permitidas, dependências inválidas ou violações de política antes que produzam side effects.

</details>

<details>
<summary><strong>5. Por que plano não é chain-of-thought?</strong></summary>

Porque o plano é um artefato operacional explícito: ações, dependências, pré-condições e estados podem ser inspecionados, validados e auditados. Ele não representa raciocínio privado oculto do modelo.

</details>

<details>
<summary><strong>6. O que o Failure Lab detecta?</strong></summary>

Ele detecta publicação posicionada antes da etapa de aprovação. A falha é de plan quality e deve ser rejeitada antes da execução.

</details>

<details>
<summary><strong>7. O que é stale plan?</strong></summary>

É um plano construído sob fatos que deixaram de ser verdadeiros. Sua estrutura pode continuar sintaticamente válida, mas suas pré-condições não refletem mais o ambiente atual.

</details>

<details>
<summary><strong>8. O que caracteriza replanning real?</strong></summary>

A mudança de estado invalida o plano v1; o sistema gera um novo artefato — plano v2 —, valida esse novo plano e executa a estratégia revisada de recuperação.

</details>

<details>
<summary><strong>9. Por que a mesma dinâmica externa?</strong></summary>

Para garantir comparação justa. Se uma arquitetura recebe informação ou eventos que a outra não recebe, diferenças de resultado podem vir do experimento, e não da arquitetura.

</details>

<details>
<summary><strong>10. O que plan_versions=2 mostra?</strong></summary>

Mostra que houve dois artefatos de plano: o original e uma versão revisada. Isso torna replanning observável em vez de inferido apenas por um texto no log.

</details>

<details>
<summary><strong>11. Quando planning é só overhead?</strong></summary>

Quando a tarefa é curta, estável, previsível e a estrutura explícita do plano não melhora sucesso, segurança, recuperação ou coordenação.

</details>

<details>
<summary><strong>12. Quando workflow é melhor?</strong></summary>

Quando a sequência e branches são conhecidos, estáveis e auditáveis, especialmente quando previsibilidade e baixo overhead são mais importantes que adaptação dinâmica.

</details>

<details>
<summary><strong>13. Quando reactive loop é suficiente?</strong></summary>

Quando basta escolher a próxima ação a partir do estado atual e não há necessidade relevante de representar dependências futuras ou manter um plano persistente.

</details>

<details>
<summary><strong>14. Que evidência justificaria planning?</strong></summary>

Por exemplo: erros evitados antes da execução, melhor recuperação após mudança de estado, coordenação de dependências complexas ou redução de ações inválidas em magnitude suficiente para compensar o overhead.

</details>

<details>
<summary><strong>15. Como a Aula 21 aplica a regra do TIL?</strong></summary>

Ela não assume que planning é superior ao reactive loop. Planning só deve permanecer quando sua estrutura explícita produz utility observável suficiente para justificar custo e complexidade adicionais.

</details>

## 10. Reprodutibilidade

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
planner determinístico
plan artifact observável
```

## 11. Síntese

```text
goal
→ decomposition
→ explicit plan
→ validation
→ execution
→ observation
→ continue / replan / stop
```

> **Planejamento é útil quando a estrutura explícita de dependências e pré-condições melhora coordenação ou segurança o suficiente para justificar seu overhead.**

---

## Continue no TIL

← **[Anterior: Aula 20 — Agentic Systems Foundations](https://www.kaggle.com/code/pedrogentil/til-20-agentic-systems-foundations)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **Próxima unidade: Agentic Systems — em preparação** →